# LLM Evaluation Framework — Phase 2

In **Phase 1** you built the core loop: get an answer, judge its **faithfulness**. Good, but a real eval framework measures more than one thing, and it *remembers* every run so you can spot when quality slips.

**Phase 2 adds four things:**
1. Two more quality judges — **answer relevance** and **correctness** (vs. your known reference answer).
2. **Cost** in dollars, computed from token usage.
3. **Latency** in seconds.
4. A **DuckDB database** that saves every run with a timestamp — this is what lets you track quality *over time* (drift), which is the whole point of the tool.

Run cells top to bottom with `Shift + Enter`. This notebook is self-contained — it repeats the setup so you can run it on its own without Phase 1 open.

## Step 1 — Setup (same as Phase 1)

Install the library, load your key from Colab Secrets (🔑 panel, secret named `ANTHROPIC_API_KEY`), and add DuckDB for storage.

In [ ]:
!pip install anthropic duckdb pandas -q
print("Installed.")

In [ ]:
from google.colab import userdata
import anthropic

API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=API_KEY)
print("Client ready.")

### Config — models AND pricing

New in Phase 2: a **pricing table**. APIs charge per *token* (a token is roughly ¾ of a word), with different prices for input vs. output tokens. To turn token counts into dollars, we need the price per token for our model.

> ⚠️ **Prices change.** The numbers below are placeholders in **dollars per 1 million tokens**. Look up the current price for your model on the provider's pricing page and update these two values. Getting this approximately right is fine — the goal is a realistic cost *signal*, not accounting.

In [ ]:
TARGET_MODEL = "claude-sonnet-4-6"
JUDGE_MODEL  = "claude-sonnet-4-6"

# Price in USD per 1,000,000 tokens. UPDATE THESE from the provider's pricing page.
PRICE_PER_M_INPUT  = 3.00    # dollars per 1M input tokens (placeholder)
PRICE_PER_M_OUTPUT = 15.00   # dollars per 1M output tokens (placeholder)

print("Config set. Remember to verify the prices above.")

## Step 2 — Load your dataset

Same dataset shape as Phase 1. The 5 sample medical items are included so this runs immediately. Notice we now actually *use* the `reference_answer` field — the correctness judge compares against it.

In [ ]:
eval_dataset = [
    {
        "id": "q001",
        "question": "What is the recommended adult dosage of amoxicillin for a mild ear infection?",
        "context": "Amoxicillin is a penicillin antibiotic used to treat bacterial infections. For a mild to moderate ear infection in adults, the usual recommended dosage is 500 mg taken orally every 12 hours for 7 days. The medication should be taken with a full glass of water and may be taken with or without food. Patients allergic to penicillin should not take amoxicillin.",
        "reference_answer": "500 mg every 12 hours for 7 days."
    },
    {
        "id": "q002",
        "question": "Who should not take amoxicillin?",
        "context": "Amoxicillin is a penicillin antibiotic used to treat bacterial infections. For a mild to moderate ear infection in adults, the usual recommended dosage is 500 mg taken orally every 12 hours for 7 days. The medication should be taken with a full glass of water and may be taken with or without food. Patients allergic to penicillin should not take amoxicillin.",
        "reference_answer": "Patients who are allergic to penicillin should not take amoxicillin."
    },
    {
        "id": "q003",
        "question": "What lifestyle changes are recommended to lower high blood pressure?",
        "context": "Hypertension, or high blood pressure, can often be managed with lifestyle changes before medication is required. Recommended changes include reducing dietary sodium to less than 1,500 mg per day, engaging in at least 150 minutes of moderate aerobic exercise per week, maintaining a healthy body weight, limiting alcohol intake, and quitting smoking. If blood pressure remains above 140/90 after three months of lifestyle changes, medication may be prescribed.",
        "reference_answer": "Reduce sodium below 1,500 mg per day, exercise at least 150 minutes per week, maintain a healthy weight, limit alcohol, and quit smoking."
    },
    {
        "id": "q004",
        "question": "When might medication be prescribed for high blood pressure?",
        "context": "Hypertension, or high blood pressure, can often be managed with lifestyle changes before medication is required. Recommended changes include reducing dietary sodium to less than 1,500 mg per day, engaging in at least 150 minutes of moderate aerobic exercise per week, maintaining a healthy body weight, limiting alcohol intake, and quitting smoking. If blood pressure remains above 140/90 after three months of lifestyle changes, medication may be prescribed.",
        "reference_answer": "If blood pressure stays above 140/90 after three months of lifestyle changes."
    },
    {
        "id": "q005",
        "question": "What are common early symptoms of type 2 diabetes?",
        "context": "Type 2 diabetes often develops gradually, and early symptoms can be easy to miss. Common early signs include increased thirst, frequent urination, unexplained fatigue, blurred vision, and slow-healing wounds. Some patients also notice increased hunger and tingling in the hands or feet. Early diagnosis through a simple blood glucose test allows for better management and can prevent complications.",
        "reference_answer": "Increased thirst, frequent urination, fatigue, blurred vision, slow-healing wounds, increased hunger, and tingling in the hands or feet."
    }
]

# To load your own instead:
# from google.colab import files
# uploaded = files.upload()
# import json
# with open('eval_dataset.json') as f:
#     eval_dataset = json.load(f)

print(f"Loaded {len(eval_dataset)} items.")

## Step 3 — Get an answer (same function, now we use the extras)

Identical to Phase 1 — but in Phase 1 we captured token usage and elapsed time and didn't use them. Now we will: usage → cost, elapsed → latency.

In [ ]:
import time

def get_answer(question, context):
    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain the answer, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

    start = time.time()
    response = client.messages.create(
        model=TARGET_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.time() - start

    answer = response.content[0].text.strip()
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    return answer, usage, elapsed

## Step 4 — Three quality judges

We now have **three** LLM-as-judge scorers, each measuring a different thing. Keeping them as separate small functions (rather than one giant judge) is deliberate: each score is independent, easy to debug, and easy to explain in an interview.

- **Faithfulness** — is the answer grounded in the context? (from Phase 1)
- **Relevance** — does the answer actually address the *question*? (An answer can be faithful to the document but still not answer what was asked.)
- **Correctness** — does the answer match the known **reference answer**? (This is why Option B — writing your own reference answers — pays off.)

All three follow the same pattern and all return *only a number* for easy parsing. We factor the shared parsing into one helper.

In [ ]:
import re

def _ask_judge_for_score(judge_prompt):
    """Send a grading prompt to the judge model and parse a 0-1 score out of it."""
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    raw = response.content[0].text.strip()
    try:
        return float(raw)
    except ValueError:
        match = re.search(r"[0-1](?:\.\d+)?", raw)
        return float(match.group()) if match else None


def judge_faithfulness(answer, context):
    prompt = f"""You are grading whether an answer is faithful to a source context.
Faithful means every claim in the answer is supported by the context, with nothing invented or contradicted.

Context:
{context}

Answer:
{answer}

Score faithfulness from 0.0 (unsupported/hallucinated) to 1.0 (fully supported).
Return ONLY the number."""
    return _ask_judge_for_score(prompt)


def judge_relevance(answer, question):
    prompt = f"""You are grading whether an answer actually addresses the question asked.
Ignore correctness of facts here; judge only whether it is on-topic and responsive.

Question:
{question}

Answer:
{answer}

Score relevance from 0.0 (ignores the question) to 1.0 (directly answers it).
Return ONLY the number."""
    return _ask_judge_for_score(prompt)


def judge_correctness(answer, reference_answer):
    prompt = f"""You are grading whether a model's answer matches a known correct reference answer.
They need not be worded identically; judge whether they agree in meaning.

Reference answer:
{reference_answer}

Model's answer:
{answer}

Score correctness from 0.0 (contradicts/misses the reference) to 1.0 (fully agrees).
Return ONLY the number."""
    return _ask_judge_for_score(prompt)

## Step 5 — Cost and latency (no LLM needed)

These two metrics are just arithmetic — no judge required.

- **Cost:** `(input_tokens × input_price) + (output_tokens × output_price)`, using our per-million pricing.
- **Latency:** we already measured elapsed time in `get_answer`.

Cost per single answer is tiny (fractions of a cent), but across thousands of daily queries it adds up fast — which is exactly the kind of thing teams want visibility into.

In [ ]:
def compute_cost(usage):
    """Convert token usage into a dollar cost using the pricing config."""
    input_cost  = (usage["input_tokens"]  / 1_000_000) * PRICE_PER_M_INPUT
    output_cost = (usage["output_tokens"] / 1_000_000) * PRICE_PER_M_OUTPUT
    return input_cost + output_cost

## Step 6 — Set up the database (DuckDB)

**Why store results at all?** Because the single most valuable thing an eval framework does is show you quality *over time*. If you only ever see the latest run, you can't tell that faithfulness quietly dropped from 0.95 to 0.78 after last week's prompt change. A database with timestamps makes drift visible.

**Why DuckDB?** It's a tiny, zero-setup database that lives in a single file — perfect for a project like this. No server, no configuration.

We create one table. Each row = one question's result in one run. A shared `run_id` + `timestamp` ties together all the rows from a single execution.

In [ ]:
import duckdb

DB_PATH = "eval_runs.db"

def init_db():
    con = duckdb.connect(DB_PATH)
    con.execute("""
        CREATE TABLE IF NOT EXISTS results (
            run_id       VARCHAR,
            timestamp    TIMESTAMP,
            question_id  VARCHAR,
            model        VARCHAR,
            faithfulness DOUBLE,
            relevance    DOUBLE,
            correctness  DOUBLE,
            cost_usd     DOUBLE,
            latency_sec  DOUBLE
        )
    """)
    con.close()
    print(f"Database ready at {DB_PATH}")

init_db()

## Step 7 — The full evaluation run

Now we tie everything together into one `run_evaluation()` function. For each item it:
1. gets an answer,
2. scores it on all three quality judges,
3. computes cost and latency,
4. saves the row to DuckDB.

Every call gets a fresh `run_id` (from the current time) so runs never mix. Run this cell to execute your first *complete* evaluation.

In [ ]:
from datetime import datetime
import uuid

def run_evaluation():
    run_id = str(uuid.uuid4())[:8]
    timestamp = datetime.now()
    con = duckdb.connect(DB_PATH)

    print(f"Starting run {run_id} at {timestamp:%Y-%m-%d %H:%M:%S}\n")

    for item in eval_dataset:
        answer, usage, elapsed = get_answer(item["question"], item["context"])

        faithfulness = judge_faithfulness(answer, item["context"])
        relevance    = judge_relevance(answer, item["question"])
        correctness  = judge_correctness(answer, item["reference_answer"])
        cost         = compute_cost(usage)

        con.execute(
            "INSERT INTO results VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            [run_id, timestamp, item["id"], TARGET_MODEL,
             faithfulness, relevance, correctness, cost, elapsed]
        )

        print(f"{item['id']}: faith={faithfulness}  rel={relevance}  "
              f"correct={correctness}  ${cost:.6f}  {elapsed:.2f}s")

    con.close()
    print(f"\nRun {run_id} saved.")
    return run_id

first_run = run_evaluation()

## Step 8 — Look at your saved results

Let's read the data back out to confirm it's stored, and see per-run averages. We use pandas just for nice display.

In [ ]:
import duckdb
con = duckdb.connect(DB_PATH)

# Per-run summary: average of each metric, plus total cost
summary = con.execute("""
    SELECT
        run_id,
        MAX(timestamp)        AS run_time,
        ROUND(AVG(faithfulness), 3) AS avg_faithfulness,
        ROUND(AVG(relevance), 3)    AS avg_relevance,
        ROUND(AVG(correctness), 3)  AS avg_correctness,
        ROUND(SUM(cost_usd), 6)     AS total_cost_usd,
        ROUND(AVG(latency_sec), 2)  AS avg_latency_sec
    FROM results
    GROUP BY run_id
    ORDER BY run_time
""").df()

con.close()
summary

## Step 9 — Prove it catches drift 🎯

This is the demo that shows the whole framework earning its keep. We'll deliberately **sabotage** the target model — tell it to ignore the context and answer from general knowledge — then run the eval again. Faithfulness and correctness should visibly **drop**, and because everything is stored, you'll see the two runs side by side.

This before/after is exactly the story to put in your README with a screenshot.

In [ ]:
# Temporarily swap in a "bad" answer function that ignores the context.
_good_get_answer = get_answer

def get_answer(question, context):
    prompt = f"Answer this question from your own general knowledge. Ignore any documents. Question: {question}"
    start = time.time()
    response = client.messages.create(
        model=TARGET_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.time() - start
    answer = response.content[0].text.strip()
    usage = {"input_tokens": response.usage.input_tokens,
             "output_tokens": response.usage.output_tokens}
    return answer, usage, elapsed

print("Running a deliberately degraded model...\n")
bad_run = run_evaluation()

# Restore the good function so later runs are normal again.
get_answer = _good_get_answer
print("\nRestored the good answer function.")

In [ ]:
# Compare the good run vs. the sabotaged run
con = duckdb.connect(DB_PATH)
comparison = con.execute("""
    SELECT
        run_id,
        ROUND(AVG(faithfulness), 3) AS avg_faithfulness,
        ROUND(AVG(correctness), 3)  AS avg_correctness
    FROM results
    GROUP BY run_id
    ORDER BY MAX(timestamp)
""").df()
con.close()

print("Notice faithfulness/correctness dropping on the sabotaged run:")
comparison

## ✅ Phase 2 complete

You now have a real evaluation framework that:
- scores answers on **faithfulness, relevance, and correctness**,
- tracks **cost** and **latency**,
- **saves every run** to a database with timestamps,
- and **detects drift** — proven with the before/after demo.

**What's next (Phase 3 → the dashboard):**
Everything is now sitting in `eval_runs.db`. The next phase reads from it and builds a **Streamlit dashboard**: big-number summary cards, a line chart of faithfulness over time (the drift chart), and a table of the worst-scoring questions. That's the visual that makes the project *land* with recruiters.

**Housekeeping:**
- The database file `eval_runs.db` lives in Colab's temporary storage and disappears when the session ends. To keep it, download it (Files panel → right-click → Download) or save to Google Drive.
- Swap in your own 30–50 item dataset before you consider this "done."
- Commit this notebook to your GitHub repo.